[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megusto0/rl-lab/blob/main/02_cross_entropy.ipynb)


# 02. Метод кросс-энтропии для CartPole

Цель ноутбука — реализовать Cross-Entropy Method для обучения стохастической политики в CartPole-v1.

**Результаты обучения:**
- строить нейронную политику с логитами действий;
- генерировать эпизоды через текущую политику;
- выбирать элитные траектории по награде;
- обучать политику классификацией элитных действий.

## Источник
Lapan M., *Deep Reinforcement Learning Hands-On*, глава 4.


In [ ]:
!pip install -q gymnasium


Импортируем Gymnasium, PyTorch и базовые библиотеки. Seed фиксируется до создания сетей и среды.


In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
except ImportError:
    pass
import torch.nn as nn
import torch.optim as optim


Политика возвращает логиты двух действий. Softmax не включается в модель, потому что функция потерь `CrossEntropyLoss` сама ожидает логиты.


In [ ]:
class PolicyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 128),
            nn.ReLU(),
            nn.Linear(128, 2),
        )

    def forward(self, x):
        return self.net(x)


Функция генерирует один эпизод. Действие сэмплируется из категориального распределения, построенного по softmax от логитов.


In [ ]:
def play_episode(env, net):
    obs, info = env.reset()
    steps, total_reward = [], 0.0
    done = False
    while not done:
        obs_v = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
        probs = torch.softmax(net(obs_v), dim=1).squeeze(0)
        action = torch.distributions.Categorical(probs).sample().item()
        next_obs, reward, terminated, truncated, info = env.step(action)
        steps.append((obs, action))
        total_reward += reward
        obs = next_obs
        done = terminated or truncated
    return steps, total_reward


На каждой итерации собираем 16 эпизодов и оставляем верхние 30 % по награде. Из шагов элитных эпизодов формируются обучающие тензоры.


In [ ]:
def generate_batch(env, net, batch_size=16):
    return [play_episode(env, net) for _ in range(batch_size)]

def filter_elites(batch, percentile=70):
    rewards = np.array([reward for _, reward in batch])
    threshold = np.percentile(rewards, percentile)
    elite_steps = [step for steps, reward in batch
                   if reward >= threshold for step in steps]
    train_obs = torch.tensor(np.array([s for s, _ in elite_steps]),
                             dtype=torch.float32)
    train_act = torch.tensor([a for _, a in elite_steps], dtype=torch.long)
    return train_obs, train_act, rewards.mean(), threshold


Обучаем сеть одной градиентной итерацией на элитных действиях. Процесс останавливается, когда средняя награда в батче достигает 199.


In [ ]:
env = gym.make("CartPole-v1")
env.reset(seed=SEED)
net = PolicyNet()
optimizer = optim.Adam(net.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()
history = []

for iteration in range(1, 101):
    batch = generate_batch(env, net)
    train_obs, train_act, mean_reward, threshold = filter_elites(batch)
    optimizer.zero_grad()
    loss = criterion(net(train_obs), train_act)
    loss.backward()
    optimizer.step()
    history.append((iteration, mean_reward, threshold, loss.item()))
    print(iteration, mean_reward, threshold, loss.item())
    if mean_reward >= 199:
        break

env.close()


Кривая средней награды по итерациям показывает, как быстро CEM концентрируется на успешных траекториях.


In [ ]:
hist = pd.DataFrame(history, columns=[
    "iteration", "mean_reward", "elite_threshold", "loss"
])
plt.figure(figsize=(8, 4))
plt.plot(hist["iteration"], hist["mean_reward"], marker="o")
plt.xlabel("Iteration")
plt.ylabel("Mean reward")
plt.title("CEM training on CartPole-v1")
plt.grid(alpha=0.3)
plt.show()


Based on Lapan M., *Deep Reinforcement Learning Hands-On*, chapter 4.


In [ ]:
os.makedirs("results", exist_ok=True)
want = [1, 10, 20, 30, int(hist["iteration"].iloc[-1])]
df_results = hist[hist["iteration"].isin(want)].drop_duplicates("iteration")
print(df_results.to_string(index=False))
df_results.to_csv("results/02_cem_training.csv", index=False)
